## エグゼクティブサマリー

物流オペレーションチームは、3つのドライバー配送ルーティング戦略（静的ベースライン、動的再ルーティング、AI最適化）を3つのデポ地域にわたって比較し、平均配送遅延（分）を応答変数とするランダム化圃場試験を計画している。PROC GLMPOWERは、想定されるセル平均の*例示*データセットを用いて、80%および90%の検出力で戦略効果を検出するために必要なドライバーの総数を解き、さらにルート間のばらつきが大きくなるにつれて達成可能な検出力がどのように低下するかを示す。

# PROC GLMPOWERによるドライバー配送ルーティング試験のサンプルサイズ設計

## エグゼクティブサマリー

物流オペレーションチームは、3つのドライバー配送ルーティング戦略――**Static**（静的）ベースライン、**Dynamic**（動的）再ルーティングシステム、**AI-Optimized**（AI最適化）プランナー――を3つのデポ地域（Northeast、Midwest、West）にわたって展開するランダム化圃場試験の開始を控えている。応答変数は平均**配送遅延（分）**である。フリート能力を試験に投入する前に、チームは「期待する運用上の改善を確実に検出するには何台のドライバーが必要か？」という問いに答えなければならない。

このノートブックは**PROC GLMPOWER**を用いて、試験の背後にある一般線形モデルに対する事前的な検出力・サンプルサイズ分析を行う。想定されるセル平均の*例示*データセットから出発し、(1) 全体の戦略効果と地域効果について80%および90%の検出力に達するために必要な総登録数を解き、(2) ルート間のばらつきが増大するにつれて達成可能な検出力がどのように低下するかを示し、(3) 登録数の意思決定を支える検出力曲線を作成する。

> **重要な結論：** 誤差標準偏差が8分であるという妥当な仮定の下では、およそ**63台のドライバー**で80%の検出力が、**83台のドライバー**で90%の検出力が得られ、ルーティング戦略の効果を検出できる――しかし遅延のばらつきが10分に近づくと、90台のドライバーを登録しても70%の検出力に届かず、緊密でよく計測されたルートの価値を裏付けている。

---
## 1. 例示計画の構築

最初のステップでは、試験のレイアウトと、ルーティング戦略×デポ地域の各組み合わせについてチームが*想定する*平均遅延を符号化する。乱数シードを固定し、無視できる程度のジッターを加えることで、平均値を現実的に見せつつ3×3のバランス構造を保つ。すべてのセルで`cell_n`の重みを1とすることで、GLMPOWERに計画がバランスしていることを伝える。

In [1]:
/* ================================================================
   想定される平均遅延の例示データセットを生成する。
   ルーティング戦略×デポ地域の設計セルごとに1行。
   ================================================================ */
データ routing_trial;
   呼出 streaminit(20260531);
   長さ routing_strategy $8 depot_region $9;
   配列 strat[3]  $8 _temporary_ ('Static' 'Dynamic' 'AIOpt');
   配列 region[3] $9 _temporary_ ('Northeast' 'Midwest' 'West');
   配列 smean[3]     _temporary_ (18.0 14.5 11.0);   /* 想定される戦略平均 */
   配列 radj[3]      _temporary_ (1.5  0.0 -1.0);    /* 地域調整（分） */
   繰返 i = 1 から 3;
      繰返 j = 1 から 3;
         routing_strategy = strat[i];
         depot_region     = region[j];
         mean_delay_min   = smean[i] + radj[j]
                            + rand('normal', 0, 0.4);
         cell_n           = 1;
         出力;
      終了;
   終了;
   削除 i j;
実行;

処理 印刷 データ=routing_trial 見出 noobs;
   変数 routing_strategy depot_region mean_delay_min cell_n;
   見出 routing_strategy="配送戦略" depot_region="デポ地域" mean_delay_min="平均遅延（分）" cell_n="セル重み";
   表題 "想定セル平均：想定される配送遅延（分）";
実行;


                                                  想定セル平均：想定される配送遅延（分）                                                   

        配送戦略          デポ地域                平均遅延（分）          セル重み
Static        Northeast              19.687507651             1
Static        Midwest               17.9871067398             1
Static        West                  16.8240210086             1
Dynamic       Northeast             15.9537535365             1
Dynamic       Midwest               14.4415169858             1
Dynamic       West                  12.8636389493             1
AIOpt         Northeast             12.6143811724             1
AIOpt         Midwest                10.893885771             1
AIOpt         West                    9.635351403             1




NOTE: DATA routing_trial


NOTE: Wrote routing_trial (9 rows, 4 columns).
NOTE: DATA elapsed:
  wall  0.00 seconds
  cpu   0.00 seconds
NOTE: PROC PRINT data=routing_trial

NOTE: PROC PRINT completed: 9 observations printed, 4 variables


---
## 2. 全体計画に対するサンプルサイズ

計画ができたところで、GLMPOWERに80%および90%の検出力で**総サンプルサイズを解く**（`NTOTAL = .`）よう求める。`MODEL`文は交互作用を含む二元配置（`routing_strategy*depot_region`）を指定し、`WEIGHT cell_n`はバランスの取れたプロファイル重みを宣言し、`STDDEV = 8`は配送遅延の想定される平均二乗誤差の平方根である。`NFRACTIONAL`は切り上げる前の正確な小数の件数を報告させる。

また、AI最適化 対 静的、動的 対 静的、いずれかの再ルーティング 対 静的という3つの計画済み`CONTRAST`比較を事前登録する――これらは試験が検証するよう設計された、運用上意味のある仮説を文書化するものである。

In [2]:
/* ================================================================
   ルーティング戦略効果とデポ地域効果について80%および90%の
   検出力に達するために必要な総ドライバー数を解く。
   ================================================================ */
処理 glmpower データ=routing_trial;
   分類 routing_strategy depot_region;
   模型 mean_delay_min = routing_strategy depot_region routing_strategy*depot_region;
   見出 routing_strategy="配送戦略" depot_region="デポ地域" mean_delay_min="平均遅延（分）";
   重み cell_n;
   CONTRAST "AI最適化 対 静的"       routing_strategy -1  0  1;
   CONTRAST "動的 対 静的"          routing_strategy -1  1  0;
   CONTRAST "再ルーティング 対 静的"  routing_strategy -2  1  1;
   POWER
      nfractional
      STDDEV = 8.0
      ALPHA  = 0.05
      NTOTAL = .
      POWER  = 0.80 0.90;
   表題 "遅延に対するルーティング戦略効果を検出するためのサンプルサイズ";
実行;


                                                  想定セル平均：想定される配送遅延（分）                                                   

The GLMPOWER Procedure


           Fixed Scenario Elements           

Item                Value                    
------------------  -------------------------
Dependent Variable  平均遅延（分）                  
Source              配送戦略                     
Source              デポ地域                     
Source              配送戦略*デポ地域                

                  Computed N Total                  

   Alpha   Std Dev   N Total     Power  Actual Power
--------  --------  --------  --------  ------------
  0.0500    8.0000        63    0.8000        0.8035
  0.0500    8.0000        83    0.9000        0.9014





NOTE: PROC GLMPOWER data=routing_trial

NOTE: PROC GLMPOWER statement used.


---
## 3. ばらつきと試験規模に対する検出力の感度

サンプルサイズの答えは、事前に正確にはわからないことが多い想定誤差標準偏差に依存する。ここでは問いを逆転させる：いくつかの候補となる総登録数（`NTOTAL = 45 90 180`）を**固定**し、もっともらしい遅延標準偏差（6、8、10分）のグリッドにわたって**達成される検出力を解く**（`POWER = .`）。得られる表は感度マップである――実際のルートのばらつきが期待より大きかった場合に、各登録計画がどれほど頑健かを示す。

In [3]:
/* ================================================================
   感度グリッド：候補となる試験規模と想定される遅延標準偏差に
   わたる達成検出力。
   ================================================================ */
処理 glmpower データ=routing_trial;
   分類 routing_strategy depot_region;
   模型 mean_delay_min = routing_strategy depot_region;
   見出 routing_strategy="配送戦略" depot_region="デポ地域" mean_delay_min="平均遅延（分）";
   重み cell_n;
   POWER
      nfractional
      STDDEV = 6.0 8.0 10.0
      ALPHA  = 0.05
      NTOTAL = 45 90 180
      POWER  = .;
   表題 "ばらつきと試験規模のシナリオにわたる達成検出力";
実行;


                                                  想定セル平均：想定される配送遅延（分）                                                   

The GLMPOWER Procedure


         Fixed Scenario Elements         

Item                Value                
------------------  ---------------------
Dependent Variable  平均遅延（分）              
Source              配送戦略                 
Source              デポ地域                 

            Computed Power            

   Alpha   Std Dev   N Total     Power
--------  --------  --------  --------
  0.0500    6.0000        45    0.8084
  0.0500    8.0000        45    0.5475
  0.0500   10.0000        45    0.3729
  0.0500    6.0000        90    0.9868
  0.0500    8.0000        90    0.8681
  0.0500   10.0000        90    0.6782
  0.0500    6.0000       180    1.0000
  0.0500    8.0000       180    0.9943
  0.0500   10.0000       180    0.9434





NOTE: PROC GLMPOWER data=routing_trial

NOTE: PROC GLMPOWER statement used.


---
## 4. 登録数の意思決定のための検出力曲線

最後に、**検出力曲線**――総登録数を30台から270台まで30台刻みで増やしたときの達成検出力――を、期待される8分の標準偏差における戦略+地域モデルについてトレースする。その登録数グリッドにわたって`POWER = .`を解くことで、曲線が「N Total」対「Power」の系列として表形式で生成され、慣例的な80%と90%の目標をそれぞれどこで超えるかを読み取ることができる。

In [4]:
/* ================================================================
   検出力曲線：総ドライバー登録数に対する達成検出力、
   期待される8分の標準偏差で30から270まで30刻みで変化させる。
   ================================================================ */
処理 glmpower データ=routing_trial;
   分類 routing_strategy depot_region;
   模型 mean_delay_min = routing_strategy depot_region;
   見出 routing_strategy="配送戦略" depot_region="デポ地域" mean_delay_min="平均遅延（分）";
   重み cell_n;
   POWER
      nfractional
      STDDEV = 8.0
      ALPHA  = 0.05
      NTOTAL = 30 60 90 120 150 180 210 240 270
      POWER  = .;
   表題 "検出力曲線：総登録ドライバー数に対する達成検出力";
実行;


                                                  想定セル平均：想定される配送遅延（分）                                                   

The GLMPOWER Procedure


         Fixed Scenario Elements         

Item                Value                
------------------  ---------------------
Dependent Variable  平均遅延（分）              
Source              配送戦略                 
Source              デポ地域                 

            Computed Power            

   Alpha   Std Dev   N Total     Power
--------  --------  --------  --------
  0.0500    8.0000        30    0.3733
  0.0500    8.0000        60    0.6887
  0.0500    8.0000        90    0.8681
  0.0500    8.0000       120    0.9500
  0.0500    8.0000       150    0.9826
  0.0500    8.0000       180    0.9943
  0.0500    8.0000       210    0.9982
  0.0500    8.0000       240    0.9995
  0.0500    8.0000       270    0.9999





NOTE: PROC GLMPOWER data=routing_trial

NOTE: PROC GLMPOWER statement used.


---
## 5. 解釈と推奨事項

この分析は、オペレーションチームに弁護可能な登録計画を与える：

- **基準となるサイズ設計（第2節）。** 誤差標準偏差を8分と仮定すると、二元配置モデル全体（戦略、地域、およびその交互作用）は**80%の検出力に63台のドライバーで**、**90%の検出力に83台のドライバーで**到達する。離脱を見込んで切り上げると、**90台のドライバー**前後の登録目標が90%のしきい値を余裕を持ってクリアする。

- **感度が重要である（第3節）。** 検出力は遅延のばらつきに非常に敏感である。90台のドライバーでは、達成検出力はSD=6でおよそ99%、SD=8でおよそ87%、SD=10でおよそ68%まで低下する。45台のドライバーによるパイロットは、ばらつきが小さい場合（SD=6でおよそ81%の検出力）にのみ十分であり、SDが10に達すると著しく検出力不足（およそ37%）となる。実務上の含意は、ばらつきを抑えるために一貫性のある、よく計測されたルートに投資することは、ドライバーを追加することと同じくらい価値があるということである。

- **検出力曲線（第4節）。** 戦略+地域モデル（交互作用項を含まない、感度分析に用いたレンズ）についてトレースすると、達成検出力は30台で0.37、60台で0.69、90台で0.87、120台で0.95まで上昇し、180台で0.99を超えて平坦化する。曲線を目標と照らし合わせると、80%の検出力はおよそ77台、90%はおよそ99台で到達する――第2節のフルモデルによるサイズ設計よりわずかに高いのは、交互作用項を除くことで同じ効果がより少ないモデル自由度に分散されるためである。

**推奨事項：** およそ**90台のドライバー**（3つのデポ地域にバランスよく配分し、ルーティング戦略ごとに30台）を登録する。これは、期待される8分の標準偏差の下でフルモデルの90%の検出力をクリアし、より保守的な縮小モデルの曲線上でもおよそ87%の検出力を維持し、1四半期の運用期間内で試験を実行できるほど小規模にとどめる。

*注：* GLMPOWERは想定された*計画*を分析するものであり、観測された結果を分析するものではない――したがって、これらの数値の信頼性は想定した平均値と標準偏差に依存する。実際の配送遅延のばらつきの実証的な推定値が得られ次第、チームはサイズ設計を見直すべきである。